In [0]:
# df = spark.read.option("multiline", True).json("/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/data/")
# df.count()

660

In [0]:
%sql
-- create schema github_stream_event_catalog.bronze_layer;

In [0]:
# %sql
# SHOW VOLUMES IN github_stream_event_catalog.bronze_layer;
# CREATE VOLUME github_stream_event_catalog.bronze_layer.raw_events_volume;

In [0]:
# Step 1: Infer schema once
static_df = spark.read.option("multiline", True).json(
    "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/data/"
)

schema = static_df.schema

# Step 2: Streaming ingestion (Auto Loader)
bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", True)
    # .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/schema"
    )
    .schema(schema)
    .load("/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/data/")
)


# Step 3: Write to Bronze table
(bronze_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/bronze"
    )
    .option("mergeSchema","true")   
    .outputMode("append")
    .trigger(availableNow=True)
    .table("github_stream_event_catalog.bronze_layer.github_events_raw")
)

In [0]:

# bronze_df.printSchema()
# schema.fields


[StructField('actor', StructType([StructField('avatar_url', StringType(), True), StructField('display_login', StringType(), True), StructField('gravatar_id', StringType(), True), StructField('id', LongType(), True), StructField('login', StringType(), True), StructField('url', StringType(), True)]), True),
 StructField('created_at', StringType(), True),
 StructField('id', StringType(), True),
 StructField('org', StructType([StructField('avatar_url', StringType(), True), StructField('gravatar_id', StringType(), True), StructField('id', LongType(), True), StructField('login', StringType(), True), StructField('url', StringType(), True)]), True),
 StructField('payload', StructType([StructField('action', StringType(), True), StructField('before', StringType(), True), StructField('description', StringType(), True), StructField('full_ref', StringType(), True), StructField('head', StringType(), True), StructField('issue', StructType([StructField('active_lock_reason', StringType(), True), Struct

In [0]:
# bronze_df.count()

In [0]:
# df.write.format("delta") \
# .mode("append") \
# .saveAsTable("github_stream_event_catalog.bronze_layer.github_events_raw")

In [0]:
# df1 = spark.read.option("multiline", True).json("/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/data/github_events_1.json")
# df1.count()

50

In [0]:
# df1.show()

+--------------------+--------------------+----------+--------------------+--------------------+------+--------------------+-----------+
|               actor|          created_at|        id|                 org|             payload|public|                repo|       type|
+--------------------+--------------------+----------+--------------------+--------------------+------+--------------------+-----------+
|{https://avatars....|2026-03-09T09:07:10Z|9185256067|                NULL|{486b3a140f3934e2...|  true|{1176025259, Pola...|  PushEvent|
|{https://avatars....|2026-03-09T09:07:10Z|9185256059|                NULL|{d6fff8a1f862d323...|  true|{1175976132, ubih...|  PushEvent|
|{https://avatars....|2026-03-09T09:07:09Z|9185255816|                NULL|{NULL, NULL, refs...|  true|{1176675446, best...|CreateEvent|
|{https://avatars....|2026-03-09T09:07:10Z|9185255818|                NULL|{6037c36f84e0de9c...|  true|{886194477, SoliS...|  PushEvent|
|{https://avatars....|2026-03-09T09:07:09